# Stage 0 — Preprocessing with SAM3

### Goal & Description

**Goal:** Instance segmentation (SAM3 → binary mask): The notebook  performs foreground-centric, re-ID–specific data preparation by utilizing zero-shot segmentation (SAM3) to isolate subjects and extract metadata.

**Input:** raw images (from several sources)

**Artifacts:**
* FiftyOne Dataset: A persistent, merged dataset (jaguar_sam3_prep) containing:
   * Original Filepaths & IDs.
   * Normalized Bounding Boxes (xywh).
  * Computed Metadata: view_main, quality_ok, sam_score.
* Physical Artifacts: A zipped archive of standardized crops and masks ready for metric learning/embedding model training.
* Export: Dataset serialized to disk (FiftyOneDataset format) for transport to the training environment.

Google AI:
* Save the RGB Crop (Standard Bounding Box) AND the Mask.
Why? DINOv2 was trained on natural images. If you feed it a cut-out jaguar on a purely black background (RGBA), the sharp edges between fur and black pixels create "fake features" that confuse the model.
Best Practice: Train on the RGB Crop. Use the Mask only for augmentation (e.g., "Randomly blur background").


### Findings

One dataset:

```
Marcela         294
Ousado          291
Kwang           177
Ti              146
Bororo          47
Alira           45
```

The other has 1-3 pictures of in total 35 jaguars


In [ ]:
import os

os.environ["PYTHONHASHSEED"] = "51"

In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas/"

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
# Install dependencies
%pip install --no-cache-dir -r requirements.txt

In [ ]:
import csv
import shutil
import numpy as np
from PIL import Image
import torch
import json
from tqdm import tqdm

import fiftyone as fo

In [ ]:
from config import DEVICE, SEED, NUM_WORKERS
from utils import ensure_dir
from preprocessing import clamp_and_pad_box, mask_to_rgba_foreground, pick_best_mask_box
from FiftyOne import FODataset

In [ ]:
FO_DATASET_NAME = "jaguar_stage0_sam3"

In [ ]:
# set paths
DRIVE_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Applied Computer Vision/Project_Scratchpad Ideas")
DATA_ROOT = DRIVE_ROOT / "datasets"

WORK_ROOT = Path("/content")
OUT_ROOT = WORK_ROOT / "out"

MASK_DIR = OUT_ROOT / "masks"
CROP_DIR = OUT_ROOT / "crops"
FG_DIR   = OUT_ROOT / "foreground_rgba"
CSV_PATH = OUT_ROOT / "metadata.csv"

EXPORT_DIR = WORK_ROOT / "jaguar_export" 

ensure_dir([MASK_DIR, CROP_DIR, FG_DIR, EXPORT_DIR])

# Load SAM3

In [ ]:
from huggingface_hub import login, whoami, hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")   # store as a Colab Secret
login(token=HF_TOKEN)                # writes token to HF cache for this runtime

print(whoami())

In [ ]:
# Clone project repository
%%capture

%cd $WORK_ROOT
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3

# Install SAM3
!pip install -e .

# Install example notebook dependencies (optional)
!pip install -e ".[notebooks]"

# Install development and training dependencies (optional)
!pip install -e ".[train,dev]"

In [ ]:
# Verify PyTorch and CUDA installation
!python -c "import torch; print(f'PyTorch: {torch.__version__}'); print(f'CUDA available: {torch.cuda.is_available()}')"

# Verify SAM3 installation
!python -c "from sam3.model_builder import build_sam3_image_model; print('SAM3 installation successful!')"

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

ckpt_path = hf_hub_download(
    repo_id="facebook/sam3",
    filename="sam3.pt",
)
print("checkpoint:", ckpt_path)

bpe_path = f"/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"

model = build_sam3_image_model(checkpoint_path=ckpt_path, bpe_path=bpe_path).to("cuda").eval()
processor = Sam3Processor(model, confidence_threshold=0.05)

# Create Masks, Foreground RGBAs and Crops

In [ ]:
def get_data(ext="jpg"):

    dir_name = "jaguar_data_subset"
    dir = DATA_ROOT / dir_name
    #shutil.copytree(DATA_ROOT / dir_name, dir, dirs_exist_ok=True)

    # collect images
    img_paths = sorted((dir / "data").glob(f"*.{ext}"))
    print("Found images:", len(img_paths))

    with open(dir / "samples.json", "r") as f:
        data = json.load(f)

    # basename -> label
    label_by_name = {
        Path(s["filepath"]).name: s["ground_truth"]["label"]
        for s in data["samples"]
    }

    return img_paths, label_by_name

In [ ]:
img_paths, label_by_name = get_data(ext="jpg")

In [ ]:
def create_crops_masks(image_paths, mask_dir, crop_dir, fg_dir, prompt="jaguar", pad_frac=0.15, hasLabel=False):
    """
    Run SAM3 on image_path, create masks, foreground rgba and crops and save all outputs.
    Returns a dict with fieldnames for FiftyOne.
    """
    rows = []

    for p in tqdm(image_paths):
      if hasLabel:
        label = label_by_name.get(p.name)
      else:
        label = p.parent.name

      image = Image.open(p).convert("RGB")
      W, H = image.size

      # SAM3
      state = processor.set_image(image)
      output = processor.set_text_prompt(state=state, prompt=prompt)

      masks = output.get("masks", None)
      boxes = output.get("boxes", None)
      scores = output.get("scores", None)

      best_mask, best_box, best_score = pick_best_mask_box(masks, boxes, scores)

      # if nothing detected, still write a row (with empty outputs)
      if best_mask is None or best_box is None:
          rows.append({
              "filepath": str(p),
              "label": label,
              "sam_score": float(best_score) if best_score is not None else None,
              "box_x1": None, "box_y1": None, "box_x2": None, "box_y2": None,
              "mask_path": None, "crop_path": None, "fg_rgba_path": None
          })
          continue

      x1, y1, x2, y2 = clamp_and_pad_box(best_box, W, H, pad_frac=pad_frac)

      # --- save mask (binary) ---
      mask_u8 = (best_mask.astype(np.uint8) * 255)
      mask_path = mask_dir / f"{label}__{p.stem}__mask.png"
      Image.fromarray(mask_u8).save(mask_path)

      # --- save crop RGB ---
      crop_rgb = image.crop((x1, y1, x2, y2))
      crop_path = crop_dir / f"{label}__{p.stem}__crop.png"
      crop_rgb.save(crop_path)

      # --- save foreground RGBA (full image) ---
      fg_rgba = mask_to_rgba_foreground(image, best_mask)
      fg_path = fg_dir / f"{label}__{p.stem}__fg.png"
      fg_rgba.save(fg_path)

      # --- derive quality signals ---
      mask_area_ratio = best_mask.sum() / best_mask.size
      quality_ok = (
        0.05 <= mask_area_ratio <= 0.60 and
        (best_score is None or best_score >= 0.3)
      )

      rows.append({
          "filepath": str(p),
          "label": label,
          "sam_score": float(best_score) if best_score is not None else None,
          "box_x1": x1, "box_y1": y1, "box_x2": x2, "box_y2": y2,
          "mask_path": str(mask_path),
          "crop_path": str(crop_path),
          "fg_rgba_path": str(fg_path),
          "crop_quality_ok": quality_ok
      })

    return rows

In [ ]:
PROMPT = "jaguar"
PAD_FRAC = 0.15

rows = create_crops_masks(
    img_paths,
    mask_dir=MASK_DIR,
    crop_dir=CROP_DIR,
    fg_dir=FG_DIR,
    prompt=PROMPT,
    pad_frac=PAD_FRAC,
    hasLabel=True
)

# Build FO Dataset

In [ ]:
fo_dataset = FODataset(
    dataset_name=FO_DATASET_NAME, 
    overwrite=True
)

In [ ]:
fo_samples = []

for r in rows:
    metadata = {
        "sam_score": r["sam_score"],
        "mask_path": r["mask_path"],
        "crop_path": r["crop_path"],
        "fg_rgba_path": r["fg_rgba_path"],
        "crop_quality_ok": r["crop_quality_ok"],
    }

    detections = []
    if r["box_x1"] is not None:
        detections.append({
            "label": "jaguar",
            "box": [r["box_x1"], r["box_y1"], r["box_x2"], r["box_y2"]],
            "score": r["sam_score"]
        })

    sample = fo_dataset.create_sample(
        filepath=r["filepath"],
        label=r["label"],
        metadata=metadata,
        detections=detections
    )
    fo_samples.append(sample)

fo_dataset.add_samples(fo_samples)

In [ ]:
session = fo_dataset.launch()
print(session.url)

# Tag Left and Right Profile

In [ ]:
# Import and load the FiftyOne Dataset from export folder
fo_dataset = FODataset.load_from_export(
    export_dir= DATA_ROOT / "jaguar_export",
    dataset_name=FO_DATASET_NAME
)
ds = fo_dataset.get_dataset()

print(ds, ds.count())
print(ds.get_field_schema().keys())

In [ ]:
def detect_orientation(image_path, prompts, threshold=0.4):
    """
    Prompts SAM 3 with competing concepts to determine orientation.
    """
    # 1. Load image as PIL (Standard for SAM 3 Research Processor)
    img_pil = Image.open(image_path).convert("RGB")
    
    # 2. Run the processor
    # Most SAM 3 research wrappers take a list of prompts and return 
    # a list of results (one per prompt)
    outputs = []
    for prompt in prompts:
        # The processor usually returns a list of masks/scores for each prompt
        # We look for the 'presence' or 'max_score' of that concept
        prediction = processor(img_pil, text_prompt=prompt)
        
        # Accessing the score: SAM 3 outputs usually contain 'scores' 
        # or 'confidences' for the detection
        if len(prediction['scores']) > 0:
            # We take the best score found for this specific prompt
            score = torch.max(prediction['scores']).item()
        else:
            score = 0.0
        outputs.append(score)

    # 3. Decision Logic
    scores = np.array(outputs)
    best_idx = np.argmax(scores)
    best_score = scores[best_idx]
    
    # Thresholding for ambiguity
    if best_score < threshold:
        return "ambiguous", best_score
    
    # Mapping: 0=Left, 1=Right, 2=Frontal (based on prompts list)
    names = ["left", "right", "frontal"]
    return names[best_idx], best_score

In [ ]:
def apply_orientation_tags(dataset, prompts, threshold):
    """
    Iterates through FiftyOne and maps SAM 3 results back to samples.
    """

    print(f"Tagging orientation for {len(dataset)} samples...")
    
    # Batch updates for speed
    with dataset.save_context() as context:
        for sample in tqdm(dataset):
            orientation, confidence = detect_orientation(sample.filepath, prompts, threshold)
            
            # Save to custom fields
            sample["orientation_sam3"] = orientation
            sample["orientation_conf"] = confidence
            
            # Tag for easy filtering in the UI
            sample.add_tag(f"orient_{orientation}")
            
            context.save(sample)

In [ ]:
prompts = [
        "left side of a jaguar", 
        "right side of a jaguar", 
        "a jaguar from the front"
    ]

fo_ds = fo_dataset.get_dataset()

apply_orientation_tags(fo_ds, prompts, threshold=0.4)


# Inspect Dataset

In [ ]:
ds = fo_dataset.dataset

# Use FiftyOne's optimized aggregation (count_values)
counts = ds.count_values("ground_truth.label")

print(f"Total Samples: {len(ds)}\n")
print(f"{'Jaguar Label':<15s} {'Count':<5s}")
print("-" * 22)

# Sorted by count (most common first)
for lab, n in sorted(counts.items(), key=lambda x: x[1], reverse=True):
    print(f"{lab:<15s} {n:<5d}")

# Export and save FO Dataset on disk

In [ ]:
fo_dataset.export_portable_dataset(EXPORT_DIR)

In [ ]:
shutil.copytree(EXPORT_DIR, DATA_ROOT / EXPORT_DIR.name, dirs_exist_ok=True)
print(f"Copied to: {DATA_ROOT / EXPORT_DIR.name}")